In [ ]:
# Interactive Figure
%matplotlib ipympl 

In [ ]:
import time
import numpy as np
from matplotlib import pyplot as plt
from qualang_tools.units import unit
u = unit(coerce_to_integer=True)
from qm.qua import *
from qm import SimulationConfig
from qualang_tools.loops import from_array
import QM

# Transmission measurement in $IQ$ plane 

In [ ]:
n_points = 2048
with program() as prog:
    i = declare(int)
    n = declare(int)
    I1 = declare(fixed)  # QUA variable for the measured 'I' quadrature
    Q1 = declare(fixed)  # QUA variable for the measured 'Q' quadrature
    I1_st = declare_stream()  # Stream for the 'I' quadrature
    Q1_st = declare_stream()  # Stream for the 'Q' quadrature
    I2 = declare(fixed)  # QUA variable for the measured 'I' quadrature
    Q2 = declare(fixed)  # QUA variable for the measured 'Q' quadrature
    I2_st = declare_stream()  # Stream for the 'I' quadrature
    Q2_st = declare_stream()  # Stream for the 'Q' quadrature

    with for_(n, 0, n < n_points, n + 1):
        # Play a weak pulse
        reset_frame("qubit")
        reset_frame("scope")
        play("pulse"*amp(0.05), "qubit")
        # Demodulate the signals to get the 'I' & 'Q' quadratures)
        measure(
            "short_readout",
            "scope",
            dual_demod.full("cos", "sin", I1),
            dual_demod.full("minus_sin", "cos", Q1),
        )
        align()
        # Play a second puilse rotated in the IQ plane
        frame_rotation_2pi(0.05, "qubit")
        play("pulse"*amp(0.05), "qubit")
        measure(
            "short_readout",
            "scope",
            dual_demod.full("cos", "sin", I2),
            dual_demod.full("minus_sin", "cos", Q2),
        )        
        # Save the 'I' & 'Q' quadratures to their respective streams
        save(I1, I1_st)
        save(Q1, Q1_st)
        save(I2, I2_st)
        save(Q2, Q2_st)

    with stream_processing():
        # Cast the data into a 1D vector, and store the results on the OPX processor
        I1_st.buffer(n_points).save("I1")
        Q1_st.buffer(n_points).save("Q1")
        I2_st.buffer(n_points).save("I2")
        Q2_st.buffer(n_points).save("Q2")


In [ ]:
# Run the code 
job = QM.Job(prog)

In [ ]:
# Plot
I1, Q1, I2, Q2 = job.get_results('I1','Q1', 'I2', 'Q2')
fig,ax=plt.subplots()
ax.plot(I1,Q1,',')
ax.plot(I2,Q2,',')
ax.set_aspect('equal')

# Exercises

## Exercise 1: Histograms
- Rotate the data in the IQ plane so that the difference between the two datasets is encoded on a single quadrature.
- Plot the histograms of the resulting quadrature.

## Exercise 2: Readout Error
- Estimate the probability of assigning an experimental point to the wrong dataset.
- Experimentally determine the minimum amplitude such that the readout error is below 5%.